# Loading libraries


In [ ]:
import pandas as pd
import numpy as np
import glob
import openpyxl

# Loading datasets

In [ ]:
var_drug_ann = pd.read_csv("/Users/ricardo/Desktop/MECD/2ndYear/Thesis/Datasets/ClinPGx/variantAnnotations/var_drug_ann.tsv", sep="\t")
var_drug_ann.head()

# Data processing


## ClinPGx data

#### **Remove "no" significant associations**


In [ ]:
var_drug_ann_clean_v1 = var_drug_ann[var_drug_ann["Significance"] != "no"]

#### **Mining data to understand non-rsID variants**


In [ ]:
var_drug_ann_clean_v2 = var_drug_ann_clean_v1[var_drug_ann_clean_v1['Variant/Haplotypes'].str.contains('rs', na=False)]

var_drug_ann_clean_v2.to_csv("/Users/ricardo/Desktop/MECD/2ndYear/Thesis/Datasets/ClinPGx/var_drug_ann_clean_v2.csv", sep="\t", index=False)


## Further cleaning the dataset
var_drug_ann_clean_v2 = var_drug_ann_clean_v2[~var_drug_ann_clean_v2['Variant/Haplotypes'].str.contains('metaboli', na=False)]

var_drug_ann_clean_v2 = var_drug_ann_clean_v2[~var_drug_ann_clean_v2['Variant/Haplotypes'].str.contains('HTTLPR short form', na=False)]

var_drug_ann_clean_v2['Variant/Haplotypes'] = var_drug_ann_clean_v2['Variant/Haplotypes'].str.replace('SLC6A4 L allele-rs25531T', 'rs25531')



In [ ]:
## Create dataset with variants with non rsID annotation
var_drug_ann_non_rs_mining = var_drug_ann_clean_v1[~var_drug_ann_clean_v1['Variant/Haplotypes'].str.contains('rs', na=False)]

## Create dataset with CYPs
var_drug_ann_cyps = var_drug_ann_non_rs_mining[var_drug_ann_non_rs_mining['Variant/Haplotypes'].str.contains('CYP', na=False)].copy()

## Create dataset with non CYPS
var_drug_ann_non_cyps = var_drug_ann_non_rs_mining[~var_drug_ann_non_rs_mining['Variant/Haplotypes'].str.contains('CYP', na=False)]

#### **Add information of about the chromosome, position, ref and alt alleles to rsID dataset**



In [ ]:
(var_drug_ann_clean_v2[['Variant Annotation ID', 'Variant/Haplotypes', 'Sentence', 'Alleles', 'Is/Is Not associated', 'Direction of effect', 'Comparison Allele(s) or Genotype(s)']]
 .to_csv("/Users/ricardo/Desktop/MECD/2ndYear/Thesis/Datasets/ClinPGx/var_drug_ann_clean_v2.csv", sep="\t", index=False))

In [ ]:
var_drug_ann_clean_v2_rs_mapped = pd.read_csv("/Users/ricardo/Desktop/MECD/2ndYear/Thesis/Datasets/ClinPGx/var_drug_ann_clean_v2_rs_mapped.csv", sep=",")

var_drug_ann_clean_v2_rs_mapped = var_drug_ann_clean_v2_rs_mapped[~var_drug_ann_clean_v2_rs_mapped['chr'].str.contains('HSCHR', na=False)]

var_drug_ann_clean_v2_rs_mapped['User_input'] = (
    var_drug_ann_clean_v2_rs_mapped['chr'].astype(str) + ":" +
    var_drug_ann_clean_v2_rs_mapped['pos'].astype('Int64').astype(str) + ":" +
    var_drug_ann_clean_v2_rs_mapped['ref'].astype(str) + ":" +
    var_drug_ann_clean_v2_rs_mapped['alt'].astype(str)
    )

var_drug_ann_clean_v2_full = pd.merge(var_drug_ann_clean_v2,
                                            var_drug_ann_clean_v2_rs_mapped[['query_id', 'User_input']],
                                            left_on="Variant/Haplotypes",
                                            right_on="query_id",
                                            how="left"
                                            )

var_drug_ann_clean_v2_full = var_drug_ann_clean_v2_full.drop(['query_id'], axis=1)


In [ ]:
def is_valid(row):
    user_input = str(row['User_input'])
    alleles = str(row['Alleles'])
    comp_alleles = str(row['Comparison Allele(s) or Genotype(s)'])
    sentence = str(row['Sentence'])

    parts = user_input.split(':')
    if len(parts) >= 4:
        ref = parts[2]
        alt = parts[3]

        if len(ref) > 1 or len(alt) > 1:
            if alt in alleles.split('/') or alt in comp_alleles.replace('+', ' ').replace('/', ' ').split():
                return True

        allowed_chars = set()
        for text in [alleles, comp_alleles]:
            if pd.notna(text) and text != 'nan':
                for char in text:
                    if char in ['A', 'C', 'G', 'T']:
                        allowed_chars.add(char)

        if len(ref) == 1 and len(alt) == 1:
            if alt in allowed_chars:
                return True

        if len(ref) > len(alt):
            if 'del' in alleles or 'del' in comp_alleles or 'del' in sentence:
                return True

        if len(ref) < len(alt):
            if 'ins' in alleles or 'ins' in comp_alleles or 'ins' in sentence:
                return True

    return False


var_drug_ann_clean_v2_full_filtered = var_drug_ann_clean_v2_full[var_drug_ann_clean_v2_full.apply(is_valid, axis=1)]



def flip_effect(effect):
    if pd.isna(effect): return effect
    effect = str(effect).strip()
    if effect == 'increased': return 'decreased'
    if effect == 'decreased': return 'increased'
    return effect


for idx, row in var_drug_ann_clean_v2_full_filtered.iterrows():
    user_input = str(row['User_input'])
    parts = user_input.split(':')

    if len(parts) >= 4:
        ref_allele = parts[2]
        causal_allele = str(row['Alleles']).strip()

        if causal_allele == ref_allele:
            comp_allele = str(row['Comparison Allele(s) or Genotype(s)']).strip()

            var_drug_ann_clean_v2_full_filtered.at[idx, 'Alleles'] = comp_allele
            var_drug_ann_clean_v2_full_filtered.at[idx, 'Comparison Allele(s) or Genotype(s)'] = causal_allele

            direction = row['Direction of effect']
            var_drug_ann_clean_v2_full_filtered.at[idx, 'Direction of effect'] = flip_effect(direction)


var_drug_ann_clean_v2_full_filtered.to_csv("/Users/ricardo/Desktop/MECD/2ndYear/Thesis/Datasets/ClinPGx/var_drug_ann_clean_v2_full_filtered.csv", sep="\t", index=False)


#### **Processing "CYP" dataset**

In [ ]:
## Edit the "CYP" dataset

def explode_df(df):
  df['Variant/Haplotypes'] = df['Variant/Haplotypes'].str.split(',')
  df_exploded = df.explode('Variant/Haplotypes')
  df_exploded['Variant/Haplotypes'] = df_exploded['Variant/Haplotypes'].str.strip()
  df_exploded = df_exploded.reset_index(drop=True)

  return df_exploded


var_drug_ann_cyps_exploded = explode_df(var_drug_ann_cyps)

var_drug_ann_cyps_exploded = var_drug_ann_cyps_exploded[var_drug_ann_cyps_exploded['Is/Is Not associated'] != "Not associated with"]

var_drug_ann_cyps_exploded['original_order'] = range(len(var_drug_ann_cyps_exploded))

var_drug_ann_cyps_exploded.to_csv("/Users/ricardo/Desktop/MECD/2ndYear/Thesis/Datasets/ClinPGx/var_drug_ann_cyps_exploded.csv", sep="\t", index=False)

var_drug_ann_cyps_exploded.head(2)

In [ ]:
## After cleaning the dataset using Gemini 3.1 Pro

var_drug_ann_cyps_exploded_cleaned = pd.read_csv("/Users/ricardo/Desktop/MECD/2ndYear/Thesis/Datasets/ClinPGx/var_drug_ann_cyps_exploded_clean.csv",
                                                 sep=";")
var_drug_ann_cyps_exploded_cleaned.head(2)

In [ ]:
var_drug_ann_cyps_exploded_filtered =pd.merge(var_drug_ann_cyps_exploded,
                                              var_drug_ann_cyps_exploded_cleaned[['original_order']],
                                              on="original_order"
                                              )



##### Proccess the *1 alleles which are treated as causal



In [ ]:
var_drug_ann_cyps_exploded_filtered_1s = var_drug_ann_cyps_exploded_filtered[var_drug_ann_cyps_exploded_filtered['Variant/Haplotypes'].str.contains('\*1$', na=False, regex=True)]

var_drug_ann_cyps_exploded_filtered_1s.to_csv("/Users/ricardo/Desktop/MECD/2ndYear/Thesis/Datasets/ClinPGx/var_drug_ann_cyps_exploded_filtered_1s.csv",
                                              sep="\t",
                                              index=False
                                              )

# Load dataframe resulting from Gemini 3.1 Pro
var_drug_ann_cyps_exploded_filtered_1s_cleaned = pd.read_csv("/Users/ricardo/Desktop/MECD/2ndYear/Thesis/Datasets/ClinPGx/var_drug_ann_cyps_exploded_filtered_1s_clean.csv",
                                                             sep=";")

var_drug_ann_cyps_exploded_filtered_1s_cleaned = var_drug_ann_cyps_exploded_filtered_1s_cleaned[~var_drug_ann_cyps_exploded_filtered_1s_cleaned['Variant/Haplotypes'].str.contains('[Placeholder]', regex=False, na=False)]

# Merge with the original "1s" dataframe to keep only the Variants that were changed
var_drug_ann_cyps_exploded_filtered_1s_processed = pd.merge(var_drug_ann_cyps_exploded_filtered_1s_cleaned,
                                                            var_drug_ann_cyps_exploded_filtered_1s,
                                                            on="original_order",
                                                            how="left"
                                                            )


var_drug_ann_cyps_exploded_filtered_1s_processed.drop(['Variant Annotation ID_x', 'Variant/Haplotypes_y', 'Gene_x', 'Drug(s)_x', 'PMID_x', 'Comparison Allele(s) or Genotype(s)_x',
                                                       'Direction of effect_y'],
                                                      axis=1,
                                                      inplace=True
                                                      )


var_drug_ann_cyps_exploded_filtered_1s_processed.columns = var_drug_ann_cyps_exploded_filtered_1s_processed.columns.str.replace('_[xy]', '', regex=True)

var_drug_ann_cyps_exploded_filtered_1s_processed = var_drug_ann_cyps_exploded_filtered_1s_processed.rename(columns={
    'Alleles': 'Comparison Allele(s) or Genotype(s)',
    'Comparison Allele(s) or Genotype(s)': 'Alleles'
    }
                                                                                                           )


var_drug_ann_cyps_exploded_filtered_1s_processed = var_drug_ann_cyps_exploded_filtered_1s_processed[var_drug_ann_cyps_exploded_filtered_1s.columns]


In [ ]:
var_drug_ann_cyps_exploded_filtered = var_drug_ann_cyps_exploded_filtered[~var_drug_ann_cyps_exploded_filtered['Variant/Haplotypes'].str.contains('\*1$', na=False, regex=True)]

var_drug_ann_cyps_exploded_filtered = pd.concat([var_drug_ann_cyps_exploded_filtered, var_drug_ann_cyps_exploded_filtered_1s_processed], axis=0)

var_drug_ann_cyps_exploded_filtered = var_drug_ann_cyps_exploded_filtered.sort_values(by=["original_order"])


##### Final "CYP" dataset

In [ ]:
final_cyp_dataset = var_drug_ann_cyps_exploded_filtered[
    ~var_drug_ann_cyps_exploded_filtered['Variant/Haplotypes'].str.contains('N', na=False, regex=False) &
    ~var_drug_ann_cyps_exploded_filtered['Variant/Haplotypes'].str.contains('metabol', na=False, regex=False) &
    ~var_drug_ann_cyps_exploded_filtered['Variant/Haplotypes'].str.contains('activity', na=False, regex=False) &
    ~var_drug_ann_cyps_exploded_filtered['Variant/Haplotypes'].str.contains('defici', na=False, regex=False) &
    ~var_drug_ann_cyps_exploded_filtered['Variant/Haplotypes'].str.contains('10x2', na=False, regex=False)
]

final_cyp_dataset.to_csv("/Users/ricardo/Desktop/MECD/2ndYear/Thesis/Datasets/ClinPGx/final_cyp_dataset.csv", sep="\t", index=False)

#### **Processing "non-CYP" dataset**

In [ ]:
var_drug_ann_non_cyps_narm = var_drug_ann_non_cyps[~var_drug_ann_non_cyps['Alleles'].isna()]

var_drug_ann_non_cyps_narm = explode_df(var_drug_ann_non_cyps_narm)

var_drug_ann_non_cyps_narm['original_order'] = range(len(var_drug_ann_non_cyps_narm))

var_drug_ann_non_cyps_narm = var_drug_ann_non_cyps_narm[~var_drug_ann_non_cyps_narm['Variant/Haplotypes'].str.contains("form", regex=False)]


var_drug_ann_non_cyps_narm.to_csv("/Users/ricardo/Desktop/MECD/2ndYear/Thesis/Datasets/ClinPGx/var_drug_ann_non_cyps_narm.csv", sep="\t", index=False)

In [ ]:
# Load dataframe resulting from Gemini 3.1 Pro
var_drug_ann_non_cyps_narm_clean = pd.read_csv("/Users/ricardo/Desktop/MECD/2ndYear/Thesis/Datasets/ClinPGx/var_drug_ann_non_cyps_narm_clean.csv",
                                               sep="\t")

In [ ]:
var_drug_ann_non_cyps_narm_clean['original_order'] = var_drug_ann_non_cyps_narm_clean['original_order'].astype(int)
var_drug_ann_non_cyps_narm['original_order'] = var_drug_ann_non_cyps_narm['original_order'].astype(int)


var_drug_ann_non_cyps_narm_processed = pd.merge(var_drug_ann_non_cyps_narm_clean,
                                                var_drug_ann_non_cyps_narm,
                                                on="original_order",
                                                how="left"
                                                )




var_drug_ann_non_cyps_narm_processed.drop(['Variant Annotation ID_x', 'Variant/Haplotypes', 'Gene_x', 'Drug(s)_x', 'PMID_x', 'Sentence_x'],
                                                      axis=1,
                                                      inplace=True
                                                      )


var_drug_ann_non_cyps_narm_processed.columns = var_drug_ann_non_cyps_narm_processed.columns.str.replace('_[xy]', '', regex=True)

var_drug_ann_non_cyps_narm_processed = var_drug_ann_non_cyps_narm_processed.rename(columns={'Variant/Haplotype': 'Variant/Haplotypes'})

var_drug_ann_non_cyps_narm_processed = var_drug_ann_non_cyps_narm_processed[var_drug_ann_non_cyps_narm.columns]



##### Proccess the *1 alleles which are treated as causal

In [ ]:
## Processing the *1 that were considered causal (after manual inspection)

# Load dataframe resulting from Gemini 3.1 Pro
var_drug_ann_non_cyps_narm_1s = pd.read_csv("/Users/ricardo/Desktop/MECD/2ndYear/Thesis/Datasets/ClinPGx/var_drug_ann_non_cyps_narm_1s.csv",
                                               sep="\t")



var_drug_ann_non_cyps_narm_1s = pd.merge(var_drug_ann_non_cyps_narm_1s,
                                         var_drug_ann_non_cyps_narm,
                                         on="original_order",
                                         how="left"
                                         )


var_drug_ann_non_cyps_narm_1s.drop(['Variant Annotation ID_x', 'Variant/Haplotypes', 'Gene_x', 'Drug(s)_x', 'PMID_x', 'Sentence_x'],
                                                      axis=1,
                                                      inplace=True
                                                      )


var_drug_ann_non_cyps_narm_1s.columns = var_drug_ann_non_cyps_narm_1s.columns.str.replace('_[xy]', '', regex=True)

var_drug_ann_non_cyps_narm_1s = var_drug_ann_non_cyps_narm_1s.rename(columns={'Variant/Haplotype': 'Variant/Haplotypes'})

var_drug_ann_non_cyps_narm_1s = var_drug_ann_non_cyps_narm_1s[var_drug_ann_non_cyps_narm.columns]

var_drug_ann_non_cyps_narm_1s['Direction of effect'] = var_drug_ann_non_cyps_narm_1s['Direction of effect'].replace({
    'increased': 'decreased',
    'decreased': 'increased'
})



var_drug_ann_non_cyps_narm_processed = var_drug_ann_non_cyps_narm_processed[
    ~var_drug_ann_non_cyps_narm_processed['original_order'].isin([67, 71, 188, 193])
]


##### Final "non-CYP" dataset

In [ ]:
# Produce a final "non-CYP" dataset
final_non_cyp_dataset = pd.concat([var_drug_ann_non_cyps_narm_processed, var_drug_ann_non_cyps_narm_1s], axis=0).sort_values(by=["original_order"])
final_non_cyp_dataset.to_csv("/Users/ricardo/Desktop/MECD/2ndYear/Thesis/Datasets/ClinPGx/final_non_cyp_dataset.csv", sep="\t", index=False)

#### **Join the "CYP" and "non-CYP" datasets**

In [ ]:
var_drug_ann_non_rs_mining_cleaned = pd.concat([final_cyp_dataset, final_non_cyp_dataset],
                                               axis=0)

var_drug_ann_non_rs_mining_cleaned = var_drug_ann_non_rs_mining_cleaned.drop(['original_order'], axis=1)

var_drug_ann_non_rs_mining_cleaned.to_csv("/Users/ricardo/Desktop/MECD/2ndYear/Thesis/Datasets/ClinPGx/final_non_rs_mining_cleaned.csv",
                                          sep="\t",
                                          index=False)

## Processing PharmVar dataset

In [ ]:
pharmvar_df_paths = glob.glob('/Users/ricardo/Desktop/MECD/2ndYear/Thesis/Datasets/PharmVar/pharmvar-6.2.22/*haplotypes*')

#Removed this dataset because its content was about rs ids
pharmvar_df_paths.remove("/Users/ricardo/Desktop/MECD/2ndYear/Thesis/Datasets/PharmVar/pharmvar-6.2.22/DPYD.NC_000001.11.haplotypes.tsv")

pharmvar_df_list = [pd.read_csv(file, sep="\t", skiprows=1) for file in pharmvar_df_paths]

pharmvar_df = pd.concat(pharmvar_df_list, ignore_index=True)


pattern = r'\*\d+\.\d+'
pharmvar_df_filtered = pharmvar_df[~pharmvar_df['Haplotype Name'].str.contains(pattern, regex=True, na=False)]
pharmvar_df_filtered = pharmvar_df_filtered.reset_index(drop=True)


pharmvar_df_filtered = pharmvar_df_filtered.dropna(subset=['rsID'])
pharmvar_df_filtered = pharmvar_df_filtered.reset_index(drop=True)


# Build dataframe with chr to merge with pharmvar_df_filtered
pharmvar_df_chr = pd.DataFrame({
    'Gene': ["CYP1A2","CYP2A13","CYP2A6","CYP2B6","CYP2C19","CYP2C8","CYP2C9","CYP2D6","CYP3A4","CYP3A5","CYP4F2","DPYD","NAT2","NUDT15","SLCO1B1"],
    'Chromosome': ["15", "19", "19", "19", "10", "10", "10", "22", "7", "7", "19", "1", "8", "13", "12"]
    }
                               )

pharmvar_df_filtered = pd.merge(pharmvar_df_filtered,
                                pharmvar_df_chr,
                                on="Gene",
                                how="left"
                                )

pharmvar_df_filtered['User_input'] = pharmvar_df_filtered['Chromosome'].astype(str) + ":" + \
                                      pharmvar_df_filtered['Variant Start'].astype(str) + ":" + \
                                      pharmvar_df_filtered['Reference Allele'].astype(str) + ":" + \
                                      pharmvar_df_filtered['Variant Allele'].astype(str)


pharmvar_df_filtered.to_csv("/Users/ricardo/Desktop/MECD/2ndYear/Thesis/Datasets/PharmVar/pharmvar_df_filtered.csv", sep="\t", index=False)

## Adding rsID to ClinPGx data

In [ ]:
## Loading and processing of allele nomenclature dataframes

#Create function for processing dataframes
def process_df_star_rs_id(df, gene_input):

    df = df.copy()

    gene = df.columns[0].replace(" ", "").split(":")[1]

    df.iloc[4] = df.iloc[4].fillna(df.iloc[0])

    df = df.drop(df.index[[0, 1, 3, 5]]).reset_index(drop=True)

    df.columns = df.iloc[1]
    df = df.drop(index=1).reset_index(drop=True)

    df['rsID'] = gene + df['rsID'].astype(str)

    df_for_merge = pd.DataFrame({
        'rsID': df.columns[1:],
        'User_input': df.iloc[0, 1:]
    }).reset_index(drop=True)

    df_for_merge['User_input'] = df_for_merge['User_input'].str.split(';')
    df_for_merge_exploded = df_for_merge.explode('User_input')
    df_for_merge_exploded['User_input'] = df_for_merge_exploded['User_input'].str.strip()
    df_for_merge_exploded = df_for_merge_exploded.reset_index(drop=True)

    if gene_input == "TPMT":
      df_for_merge_exploded['User_input'] = df_for_merge_exploded['User_input'].str.replace(r'^g\.(\d+)([ACGT])>([ACGT])$',
                                                                    r'6:\1:\2:\3',
                                                                    regex=True
                                                                    )
    else:
      df_for_merge_exploded['User_input'] = df_for_merge_exploded['User_input'].str.replace(r'^g\.(\d+)([ACGT])>([ACGT])$',
                                                                    r'2:\1:\2:\3',
                                                                    regex=True
                                                                    )

    df = df.drop(df.index[[0]]).reset_index(drop=True)
    df = pd.melt(df, id_vars=['rsID'], var_name='rs_id', value_name='allele')
    df = df.merge(df_for_merge_exploded, left_on='rs_id', right_on="rsID", how='left')
    df = df.sort_values(by=['rsID_x']).dropna().reset_index(drop=True)

    df = df.drop(columns=['allele', 'rsID_y']).rename(columns={'rsID_x': 'Haplotype Name',
                                                     'rs_id': 'rsID'})

    return df



tptm_star_alleles_rs = pd.read_excel("/Users/ricardo/Desktop/MECD/2ndYear/Thesis/Datasets/PharmVar/TPMT_allele_definition_table.xlsx")
ugt1a1_star_alleles_rs = pd.read_excel("/Users/ricardo/Desktop/MECD/2ndYear/Thesis/Datasets/PharmVar/UGT1A1_allele_definition_table.xlsx")

tptm_star_alleles_rs = process_df_star_rs_id(tptm_star_alleles_rs, gene_input="TPMT")
ugt1a1_star_alleles_rs = process_df_star_rs_id(ugt1a1_star_alleles_rs, gene_input="UGT1A1")



## Process the remaining UGT1 genes variants (the formar of the Excel file is different from the TPMT and UGT1A1 files)
ugt1a3_star_alleles_rs = pd.read_excel("/Users/ricardo/Desktop/MECD/2ndYear/Thesis/Datasets/PharmVar/UGT1A3_allele_definition_table.xlsx")
ugt1a4_star_alleles_rs = pd.read_excel("/Users/ricardo/Desktop/MECD/2ndYear/Thesis/Datasets/PharmVar/UGT1A4_allele_definition_table.xlsx")
ugt1a6_star_alleles_rs = pd.read_excel("/Users/ricardo/Desktop/MECD/2ndYear/Thesis/Datasets/PharmVar/UGT1A6_allele_definition_table.xlsx")
ugt1a8_star_alleles_rs = pd.read_excel("/Users/ricardo/Desktop/MECD/2ndYear/Thesis/Datasets/PharmVar/UGT1A8_allele_definition_table.xlsx")
ugt1a9_star_alleles_rs = pd.read_excel("/Users/ricardo/Desktop/MECD/2ndYear/Thesis/Datasets/PharmVar/UGT1A9_allele_definition_table.xlsx")
ugt2b15_star_alleles_rs = pd.read_excel("/Users/ricardo/Desktop/MECD/2ndYear/Thesis/Datasets/PharmVar/UGT2B15_allele_definition_table.xlsx")


ugt1_df_list = [ugt1a3_star_alleles_rs, ugt1a4_star_alleles_rs, ugt1a6_star_alleles_rs,
                ugt1a8_star_alleles_rs, ugt1a9_star_alleles_rs, ugt2b15_star_alleles_rs
                ]


# Create function to process the remaining UGT1 genes variants
def process_df_star_rs_id_ugt1(df):
    t = df.copy()

    t = t.drop(t.index[[0]]).reset_index(drop=True)

    t.columns = t.iloc[1]

    t = t.drop(['Haplotype ID', 'Accession ID', 'Tag(s)'], axis=1, errors='ignore')
    cols_to_drop = [
    col for col in t.columns
    if t[col].str.contains('PMID', na=False).any()
    ]
    t = t.drop(cols_to_drop, axis=1)

    t_for_merge = pd.DataFrame({
        'rsID': t.columns[1:],
        'User_input': t.iloc[0, 1:]
    }).reset_index(drop=True)

    t = t[2:].reset_index(drop=True)

    gene = t.columns[0]
    t.iloc[:,0] = gene + t[gene].astype(str)

    t.iloc[1:, 1:] = t.iloc[1:, 1:].mask(t.iloc[1:, 1:] == t.iloc[0, 1:], np.nan)

    t = pd.melt(t, id_vars=[gene], var_name='rs_id', value_name='allele')

    t = t.sort_values(by=[gene]).dropna().reset_index(drop=True)

    t = t.drop(columns=['allele']).rename(columns={gene: 'Haplotype Name',
                                                        'rs_id': 'rsID'})

    t = t.merge(t_for_merge, on='rsID', how='left')

    if gene == "UGT1A3":
      t['User_input'] = t['User_input'].str.replace(r'^g\.(\d+)([ACGT])>([ACGT])$',
                                                                    r'2:\1:\2:\3',
                                                                    regex=True
                                                                    )
    elif gene == "UGT1A4":
      t['User_input'] = t['User_input'].str.replace(r'^g\.(\d+)([ACGT])>([ACGT])$',
                                                                    r'2:\1:\2:\3',
                                                                    regex=True
                                                                    )
    elif gene == "UGT1A6":
      t['User_input'] = t['User_input'].str.replace(r'^g\.(\d+)([ACGT])>([ACGT])$',
                                                                    r'2:\1:\2:\3',
                                                                    regex=True
                                                                    )
    elif gene == "UGT1A8":
      t['User_input'] = t['User_input'].str.replace(r'^g\.(\d+)([ACGT])>([ACGT])$',
                                                                    r'2:\1:\2:\3',
                                                                    regex=True
                                                                    )
    elif gene == "UGT1A9":
      t['User_input'] = t['User_input'].str.replace(r'^g\.(\d+)([ACGT])>([ACGT])$',
                                                                    r'2:\1:\2:\3',
                                                                    regex=True
                                                                    )
    else:
      t['User_input'] = t['User_input'].str.replace(r'^g\.(\d+)([ACGT])>([ACGT])$',
                                                                    r'4:\1:\2:\3',
                                                                    regex=True
                                                                    )

    return t


processed_list = [process_df_star_rs_id_ugt1(df) for df in ugt1_df_list]

(ugt1a3_star_alleles_rs,
 ugt1a4_star_alleles_rs,
 ugt1a6_star_alleles_rs,
 ugt1a8_star_alleles_rs,
 ugt1a9_star_alleles_rs,
 ugt2b15_star_alleles_rs) = processed_list



In [ ]:
## Merging the PharmVar and the ClinPGx datasets to add the rs IDs
pharmvar_complete = (
    pd.concat([
        pharmvar_df_filtered[['Haplotype Name', 'rsID', 'User_input']],
        tptm_star_alleles_rs,
        ugt1a1_star_alleles_rs,
        ugt1a3_star_alleles_rs,
        ugt1a4_star_alleles_rs,
        ugt1a6_star_alleles_rs,
        ugt1a8_star_alleles_rs,
        ugt1a9_star_alleles_rs,
        ugt2b15_star_alleles_rs,
        pd.DataFrame({
            'Haplotype Name': ['UGT1A1*60', 'NUDT15*2', 'NUDT15*2', 'UGT2B7*2', 'G6PD variant', 'G6PD variant',
                               'DPYD*7', 'DPYD c.2846A>T', 'DPYD*13', 'DPYD*2A', 'CFTR variant'],
            'rsID': ['rs4124874', 'rs116855232', 'rs746071566', 'rs7439366', 'rs1050828', 'rs1050829',
                     'rs72549309', 'rs67376798', 'rs55886062', 'rs3918290', 'rs75039782'],
            'User_input': ['2:233757013:T:G', '13:48045719:C:T', '13:48037796:GGAGTCG:G', '4:69098620:T:C', 'X:154536002:C:T', 'X:154535277:T:C',
                           '1:97740411:ATGA:-', '1:97082391:T:A', '1:97515787:A:C', '1:97450058:C:T', '7:117639961:C:T']
        }),
    ])
    .sort_values(by=['Haplotype Name'])
    .reset_index(drop=True)
)



var_drug_ann_non_rs_mining_cleaned['original_order'] = range(len(var_drug_ann_non_rs_mining_cleaned))

var_drug_ann_non_rs_withrs = pd.merge(var_drug_ann_non_rs_mining_cleaned,
                                      pharmvar_complete,
                                      how="left",
                                      left_on="Variant/Haplotypes",
                                      right_on="Haplotype Name"
                                      )



In [ ]:
## Filter after manual inspection of the var_drug_ann_non_rs_withrs dataset
# Remove entries in a CYP gene without rs id
is_cyp = var_drug_ann_non_rs_withrs['Gene'].str.contains('CYP', na=False)
is_missing_user_input = var_drug_ann_non_rs_withrs['User_input'].isna() | (var_drug_ann_non_rs_withrs['User_input'] == '-') | (var_drug_ann_non_rs_withrs['User_input'] == '')
var_drug_ann_non_rs_withrs = var_drug_ann_non_rs_withrs[~(is_cyp & is_missing_user_input)]

#Remove HLAs
var_drug_ann_non_rs_withrs = var_drug_ann_non_rs_withrs[~var_drug_ann_non_rs_withrs['Gene'].str.contains('HLA', na=False)]


# Remove empty values
var_drug_ann_non_rs_withrs = var_drug_ann_non_rs_withrs[
    ~((var_drug_ann_non_rs_withrs['User_input'].isna()) |
      (var_drug_ann_non_rs_withrs['User_input'] == '-') |
      (var_drug_ann_non_rs_withrs['User_input'] == ''))
]



var_drug_ann_non_rs_withrs = var_drug_ann_non_rs_withrs.reset_index(drop=True)


var_drug_ann_non_rs_withrs.to_csv("/Users/ricardo/Desktop/MECD/2ndYear/Thesis/Datasets/ClinPGx/var_drug_ann_non_rs_withrs.csv",
                                          sep="\t",
                                          index=False)

## Produce the final ClinPGx processed dataset

In [ ]:
## Edit the var_drug_ann_non_rs_withrs to match the structure of the var_drug_ann_clean_v2 dataframe
var_drug_ann_non_rs_final = var_drug_ann_non_rs_withrs.copy()

var_drug_ann_non_rs_final = var_drug_ann_non_rs_final.drop(
    columns=['Haplotype Name', 'original_order']
)

var_drug_ann_non_rs_final['Variant/Haplotypes'] = var_drug_ann_non_rs_final['rsID']
var_drug_ann_non_rs_final = var_drug_ann_non_rs_final.drop(columns=['rsID'])

var_drug_ann_non_rs_final = var_drug_ann_non_rs_final[var_drug_ann_clean_v2_full_filtered.columns]

var_drug_ann_non_rs_final

In [ ]:
final_clinpgx_processed_dataset = pd.concat([var_drug_ann_clean_v2_full_filtered, var_drug_ann_non_rs_final], axis=0)

final_clinpgx_processed_dataset['Drug(s)'] = final_clinpgx_processed_dataset['Drug(s)'].str.split(r',\s+', regex=True)
final_clinpgx_processed_dataset = final_clinpgx_processed_dataset.explode('Drug(s)')
final_clinpgx_processed_dataset['Drug(s)'] = final_clinpgx_processed_dataset['Drug(s)'].str.strip()
final_clinpgx_processed_dataset = final_clinpgx_processed_dataset.reset_index(drop=True)

final_clinpgx_processed_dataset = final_clinpgx_processed_dataset.drop_duplicates()


final_clinpgx_processed_dataset['User_input'] = (
    final_clinpgx_processed_dataset['User_input'].astype(str) + ":" +
    final_clinpgx_processed_dataset['Gene'].astype(str)
    )


final_clinpgx_processed_dataset.to_csv("/Users/ricardo/Desktop/MECD/2ndYear/Thesis/Datasets/ClinPGx/final_clinpgx_processed_dataset.csv",
                                       sep="\t",
                                       index=False,
                                       na_rep="NA")
